# JSpaceDoNOT — GSM8K, MATH-500, RACE-H

Run from the repository root in the environment described in README.md. Each task uses the uploaded settings (256 TRAIN questions, seed 2026, four folds, all TEST questions). Default schedule is **wait** as requested; set `SCHEDULE="token"` for the uploaded notebooks' every-256-token runs. Preparation commands and J++ fitting are documented in README; do those once before these experiments.


In [ ]:
from pathlib import Path
from dataclasses import replace
from jspace_donot import ExperimentConfig, ExperimentResult, run_experiment, required_paths

ROOT = Path(".").resolve()
SCHEDULE = "wait"                 # "wait" | "token"
EVERY_N_TOKENS = 256               # token mode only
CACHE_MODE = "resume"              # "resume" | "read_only" | "off"
CACHE_ROOT = None                  # None: original cache paths; or Path("/your/cache")
SOURCE_MODE = "auto"               # GSM8K: "historical" for exact existing-CoT replay
LENS_PATH = None                   # or Path(".../qwen3_4b_jpp_lens.pt")

common = ExperimentConfig(
    dataset="gsm8k", root=ROOT,
    schedule=SCHEDULE, every_n_tokens=EVERY_N_TOKENS,
    wait_markers=("Wait",), min_stop_tokens=256,
    cache_mode=CACHE_MODE, cache_root=CACHE_ROOT,
    source_mode=SOURCE_MODE, lens_path=LENS_PATH,
)
configs = {task: replace(common, dataset=task) for task in ("gsm8k", "math500", "raceh")}
results = {}
for task, config in configs.items():
    print(task, required_paths(config) or "Prerequisites found")


## Optional one-time preparation

Set only the steps you need to `True`. Reuse your completed J++ checkpoint whenever available. `FIT_JPP` is an expensive GPU job. Downloads record revisions; existing different dataset files are preserved and produce an error.


In [ ]:
import subprocess, sys
SETUP_JPP = False
DOWNLOAD_MODEL = False
PREPARE_DATA = False
FIT_JPP = False

steps = []
if SETUP_JPP: steps.append("setup_jpp")
if DOWNLOAD_MODEL: steps.append("download_model")
if PREPARE_DATA: steps += ["prepare_gsm8k", "prepare_math500", "prepare_raceh"]
if FIT_JPP: steps.append("fit_jpp")
for step in steps:
    subprocess.run([sys.executable, str(ROOT / "scripts" / f"{step}.py"), "--root", str(ROOT)], check=True)


## GSM8K

The method runs in a worker process and releases its GPU memory on completion.


In [ ]:
results["gsm8k"] = run_experiment(configs["gsm8k"])
print(results["gsm8k"].output)
display(results["gsm8k"].summary)


## MATH-500

The method runs in a worker process and releases its GPU memory on completion.


In [ ]:
results["math500"] = run_experiment(configs["math500"])
print(results["math500"].output)
display(results["math500"].summary)


## RACE-H

The method runs in a worker process and releases its GPU memory on completion.


In [ ]:
results["raceh"] = run_experiment(configs["raceh"])
print(results["raceh"].output)
display(results["raceh"].summary)


## Interpret fitted LR / MLP (CPU, no retraining)

This reads the actual heads and features saved by the chosen run. After a restart, set `RUN_DIR` to a completed output folder. Other datasets use the same interpretation API.


In [ ]:
from jspace_donot.interpretation import interpret_run, StopInterpreter

RUN_DIR = results["gsm8k"].output   # or Path("jlens_outputs/qwen3_4b/<completed-run>")
INTERPRET_OUTPUT = interpret_run(
    RUN_DIR, linear_C=0.1, mlp_C=1.0, cutoff=0.85,
    max_ig_examples=300, ig_steps=48, find_cases=False,
)
print(INTERPRET_OUTPUT)


## Optional case search and raw-score intervention

This searches the predictor's first STOP per TEST question and replaces complete candidate columns with TRAIN medians. It never changes Qwen activations or generates an answer. A case search can legitimately return no cases.


In [ ]:
RUN_CASE_STUDIES = False
if RUN_CASE_STUDIES:
    explainer = StopInterpreter.load(RUN_DIR)
    cases = explainer.find_stop_flips(cutoff=0.85)
    display(cases.head(15))
    if not cases.empty:
        case = cases.iloc[0]
        raw, context = explainer.checkpoint(case.question_id, int(case.step))
        print(context.get("question"))
        print("Candidate tokens:", context.get("token_strings"))
        print("CoT prefix:", context.get("cot_prefix"))
        print("Original forced-answer outcome:", context.get("outcome"))
        table, _ = explainer.group_ablation(case.question_id, int(case.step),
            groups={"L3 ranks 1+2": [8,9], "L4 ranks 1+2": [12,13], "Both": [8,9,12,13]}, cutoff=.85)
        display(table)
        sweep = explainer.sweep(case.question_id, int(case.step), indices=[8,9])
        sweep.to_csv(INTERPRET_OUTPUT / "case_score_sweep.csv", index=False)
        from jspace_donot.interpretation_plots import plot_score_sweep
        plot_score_sweep(sweep, feature_index=8, output=INTERPRET_OUTPUT / "case_score_sweep", cutoff=.85)
